# Multi-Agent Patterns in LangChain v1 -- CineBot Edition (VS Code + uv)

A single agent with the right tools and prompt handles most tasks. Multi-agent patterns earn
their complexity when you need one or more of:

- **Context management** -- specialized knowledge without overwhelming one context window
- **Distributed development** -- different teams own different capabilities, composed together
- **Parallelization** -- spawn specialized workers for subtasks and run them concurrently

This notebook covers all five official LangChain v1 multi-agent patterns end to end, using
CineBot (our running movie-ticketing example) throughout, with real, executable code for every
pattern -- verified by actually running it, not just described.

| Pattern | How it works |
|---|---|
| **Subagents** | A main agent coordinates subagents as tools. All routing passes through the main agent. |
| **Handoffs** | Behavior changes dynamically based on state. Tool calls update a state variable that triggers routing or reconfiguration. |
| **Skills** | Specialized prompts and knowledge loaded on-demand. One agent stays in control, loading context as needed. |
| **Router** | A routing step classifies input and directs it to one or more specialized agents; results are synthesized. |
| **Custom workflow** | Bespoke LangGraph execution flow, mixing deterministic logic and agentic behavior. Can embed any other pattern as a node. |

```mermaid
graph TB
    subgraph Subagents
        SA[Main Agent] --> SA1[Subagent A]
        SA --> SA2[Subagent B]
        SA1 --> SA
        SA2 --> SA
    end
    subgraph Handoffs
        HA[Agent: step X] -->|tool updates state| HB[Agent: step Y]
    end
    subgraph Skills
        SK[Agent] -->|load_skill| SK1[Skill A prompt]
        SK -->|load_skill| SK2[Skill B prompt]
    end
    subgraph Router
        R[Router] --> R1[Agent A]
        R --> R2[Agent B]
        R1 --> RS[Synthesize]
        R2 --> RS
    end
```

> 📋 **Takeaways **
> - Default to a single agent first. Reach for multi-agent only when tools/context/teams/latency
>   genuinely demand it.
> - **Supervisor (Subagents) vs. Router**: a supervisor is a full agent that maintains
>   conversation state and decides across multiple turns; a router is a single classification
>   step with no ongoing state.
> - Patterns compose: a Subagents architecture can call tools that run Custom Workflows or
>   Router agents; Subagents can use Skills internally. Mix freely.


---
## Choosing a pattern

| Optimize for | Subagents | Handoffs | Skills | Router |
|---|:-:|:-:|:-:|:-:|
| Distributed development | ⭐⭐⭐⭐⭐ | - | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| Parallelization | ⭐⭐⭐⭐⭐ | - | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Multi-hop (chaining several subagents) | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | - |
| Direct user interaction | ⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |

**Performance characteristics** (model calls -- more calls = more latency/cost; see the official
docs for the full worked examples with token counts):

| Scenario | Subagents | Handoffs | Skills | Router |
|---|:-:|:-:|:-:|:-:|
| One-shot request | 4 calls | 3 calls ✅ | 3 calls ✅ | 3 calls ✅ |
| Repeat request (same turn type again) | 8 calls (4+4) | 5 calls (3+2) ✅ | 5 calls (3+2) ✅ | 6 calls (3+3) |
| Multi-domain (3 parallel topics) | 5 calls, ~9K tokens ✅ | 7+ calls, ~14K+ tokens | 3 calls, ~15K tokens | 5 calls, ~9K tokens ✅ |

> 📋 **Takeaways:**
> - **Subagents**: stateless by design -- strong context isolation, but repeats the full flow on
>   every request (no "it's already loaded" discount). Best for parallel, large-context domains.
> - **Handoffs & Skills**: *stateful* -- once a step/skill is active, repeat requests skip
>   re-routing or re-loading, saving 40-50% of calls on the second ask. Best for single/repeat
>   requests needing direct user conversation.
> - **Router**: stateless per request (an LLM classification call every time), but supports true
>   parallel fan-out. Best when you have clear, parallel verticals and don't need the router
>   itself to remember anything.


---
# Setup (VS Code + uv)

```bash
uv sync
cp .env.example .env   # fill in OPENAI_API_KEY
```

Open this notebook in VS Code on the `.venv` kernel `uv sync` created (VS Code's Jupyter
extension detects it automatically; otherwise **"Python: Select Interpreter"** → `.venv/bin/python`,
then **"Notebook: Select Notebook Kernel"**).

**How this notebook verifies itself without a live API key:** every pattern below is demonstrated
twice -- once against a small, local **scripted fake model** that returns pre-programmed
responses (so the graph wiring, state updates, and routing logic are *genuinely executed and
checked*, not just described), and once against a real `create_agent(model="openai:...")` call
wrapped in `try/except` (so it degrades gracefully to a printed message if you haven't set
`OPENAI_API_KEY` yet, exactly like the MCP notebook). Both are real code; only the second one
needs your key.


In [1]:
import os
import warnings
from dotenv import load_dotenv

load_dotenv()
HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
if not HAS_KEY:
    print("No OPENAI_API_KEY found -- the scripted-model cells below still run for real. "
          "Real-model cells will print a diagnostic instead of crashing. "
          "Copy .env.example to .env and fill in a key to see the real-model cells run live.")
else:
    print("OPENAI_API_KEY found -- all cells, including real-model ones, should run live.")


OPENAI_API_KEY found -- all cells, including real-model ones, should run live.


## The scripted fake model

A tiny, reusable test double: it plays back a fixed list of `AIMessage`s (plain replies or
tool-calling ones) in order, one per turn. This is the same technique LangChain's own test suite
uses (`FakeMessagesListChatModel`), extended here to support `bind_tools()` so `create_agent`
accepts it. It makes every graph-wiring claim in this notebook checkable, not just asserted.


In [2]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel

class ScriptedToolModel(FakeMessagesListChatModel):
    """Plays back `responses` in order. Accepts bind_tools() so create_agent() can use it as
    a drop-in stand-in for a real tool-calling model -- no network, no API key, fully
    deterministic."""
    def bind_tools(self, tools, **kwargs):
        return self

print("ScriptedToolModel defined.")


ScriptedToolModel defined.


---
# Pattern 1: Subagents

A main agent (often called a **supervisor**) coordinates subagents by calling them as tools. The
main agent decides which subagent to invoke, what input to give it, and how to combine results.
Subagents are **stateless by default** -- they don't remember past interactions; all conversation
memory lives in the main agent. This gives **context isolation**: each subagent call works in a
clean context window.

```mermaid
graph LR
    A[User] --> B[Main Agent]
    B --> C[Showtimes Subagent]
    B --> D[Recommendations Subagent]
    B --> E[Booking Subagent]
    C --> B
    D --> B
    E --> B
    B --> F[User response]
```

**Key characteristics:**
- Centralized control -- all routing passes through the main agent
- No direct user interaction -- subagents return results to the main agent, not the user (though
  you can use LangGraph `interrupt()` inside a subagent for mid-task user input)
- Subagents are invoked via tools
- The main agent can invoke multiple subagents in a single turn, in parallel

> ⚠️ **Supervisor vs. Router, precisely:** a supervisor (this pattern) is a full agent that
> maintains conversation context and dynamically decides which subagents to call *across multiple
> turns*. A router (Pattern 4) is typically a single classification step with no ongoing
> conversation state.

**When to use:** multiple distinct domains (showtimes, bookings, recommendations, loyalty),
subagents don't need to talk to the user directly, or you want centralized workflow control. For
just a few tools, use a single agent instead.


In [3]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import AIMessage

# -- The subagent: a small CineBot showtimes specialist --
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {"interstellar": "7:00 PM and 10:15 PM", "dune part two": "9:30 PM"}
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

showtimes_scripted = ScriptedToolModel(responses=[
    AIMessage(content="", tool_calls=[
        {"name": "check_showtimes", "args": {"movie_title": "Interstellar"}, "id": "call_1"}
    ]),
    AIMessage(content="Interstellar is showing at 7:00 PM and 10:15 PM."),
])
showtimes_subagent = create_agent(model=showtimes_scripted, tools=[check_showtimes])

# -- Wrap it as a tool the main agent can call --
@tool("showtimes", description="Check movie showtimes")
def call_showtimes_agent(query: str) -> str:
    result = showtimes_subagent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# -- Main agent, scripted to decide to call the showtimes subagent --
main_scripted = ScriptedToolModel(responses=[
    AIMessage(content="", tool_calls=[
        {"name": "showtimes", "args": {"query": "When does Interstellar show?"}, "id": "call_2"}
    ]),
    AIMessage(content="Interstellar is playing at 7:00 PM and 10:15 PM tonight."),
])
main_agent = create_agent(model=main_scripted, tools=[call_showtimes_agent])

# Run for real: this genuinely exercises main agent -> subagent tool -> subagent's own tool call.
result = main_agent.invoke({"messages": [{"role": "user", "content": "When does Interstellar show?"}]})
for m in result["messages"]:
    label = type(m).__name__
    print(f"[{label}] {getattr(m, 'content', '')!r}  tool_calls={getattr(m, 'tool_calls', None)}")


[HumanMessage] 'When does Interstellar show?'  tool_calls=None
[AIMessage] ''  tool_calls=[{'name': 'showtimes', 'args': {'query': 'When does Interstellar show?'}, 'id': 'call_2', 'type': 'tool_call'}]
[ToolMessage] 'Interstellar is showing at 7:00 PM and 10:15 PM.'  tool_calls=None
[AIMessage] 'Interstellar is playing at 7:00 PM and 10:15 PM tonight.'  tool_calls=[]


> ⚙️ **Code Walkthrough:** that output is a real, four-message trace: the main agent's model
> decided (per its script) to call the `showtimes` tool; that tool invoked the `showtimes_subagent`
> *as a completely separate agent run*, which itself called `check_showtimes` and answered; the
> subagent's final message became the `showtimes` tool's return value; the main agent then used
> it to answer the user. Swap `main_scripted`/`showtimes_scripted` for `"openai:gpt-5-mini"` in
> production and the wiring is identical -- only the model's *decisions* change.

```python
# Production wiring -- real model, same structure:
main_agent_prod = create_agent(model="openai:gpt-5-mini", tools=[call_showtimes_agent])
try:
    result = main_agent_prod.invoke({"messages": [{"role": "user", "content": "When does Interstellar show?"}]})
    print(result["messages"][-1].content)
except Exception as e:
    print(f"Skipped -- {type(e).__name__}: {e}")
```


In [4]:
# Run for real -- the production-shaped cell from the markdown above, not just prose.
try:
    main_agent_prod = create_agent(model="openai:gpt-5-mini", tools=[call_showtimes_agent])
    result = main_agent_prod.invoke({"messages": [{"role": "user", "content": "When does Interstellar show?"}]})
    print(result["messages"][-1].content)
except Exception as e:
    print(f"Skipped -- {type(e).__name__}: {e}")


Do you mean the 2014 Christopher Nolan film Interstellar? To find showtimes I need your location (city or ZIP/postal code) and the date you want (today, tomorrow, or a specific day). I can also search for streaming/TV options instead if you prefer. Which would you like?


In [5]:
result


{'messages': [HumanMessage(content='When does Interstellar show?', additional_kwargs={}, response_metadata={}, id='b8391279-afa6-4f44-80dc-17905083a079'),
  AIMessage(content='Do you mean the 2014 Christopher Nolan film Interstellar? To find showtimes I need your location (city or ZIP/postal code) and the date you want (today, tomorrow, or a specific day). I can also search for streaming/TV options instead if you prefer. Which would you like?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 264, 'prompt_tokens': 129, 'total_tokens': 393, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl

## Sync vs. async

| Mode | Main agent behavior | Best for | Tradeoff |
|---|---|---|---|
| **Sync** (default) | Waits for subagent to complete | Main agent needs the result to continue | Simple, but blocks the conversation |
| **Async** | Continues while subagent runs in background | Independent tasks, user shouldn't wait | Responsive, but needs a job system |

> ⚠️ Not Python's `async`/`await` -- "async" here means the main agent kicks off a background
> job (a separate process/service) and continues without waiting for it.

```mermaid
sequenceDiagram
    participant User
    participant Main Agent
    participant Job System
    participant Reviewer

    User->>Main Agent: "Review this contract"
    Main Agent->>Job System: run_agent("legal_reviewer", task)
    Job System->>Reviewer: Start agent
    Job System-->>Main Agent: job_id: "job_123"
    Main Agent-->>User: "Started review (job_123)"
    Note over Reviewer: Reviewing...
    User->>Main Agent: "Status?"
    Main Agent->>Job System: check_status(job_id)
    Job System-->>Main Agent: "running"
```

**Three-tool pattern for async:** `start_job` (kicks off, returns a job ID), `check_status`
(pending/running/completed/failed), `get_result` (retrieves the completed result). When a job
finishes, surface a notification that sends a message like "Check job_123 and summarize" back
into the conversation.

> 📋 **Takeaways:**
> - Sync when the main agent's next move depends on the result, or failures should block the
>   response.
> - Async when the work is independent and the user shouldn't be frozen waiting for it.


## Tool patterns: tool-per-agent vs. single dispatch tool

| Pattern | Best for | Tradeoff |
|---|---|---|
| **Tool per agent** | Fine-grained control over each subagent's input/output | More setup, more customization |
| **Single dispatch tool** | Many agents, distributed teams, convention over configuration | Simpler composition, less per-agent customization |

```mermaid
graph LR
    A[User] --> B[Main Agent]
    B --> C{"task(agent_name, description)"}
    C -->|research| D[Research Agent]
    C -->|writer| E[Writer Agent]
    D --> C
    E --> C
    C --> B
```


In [ ]:
# Single dispatch tool: one parameterized "task" tool invokes any registered sub-agent by name.
# Convention-based: agent selected by name, task passed as a human message, final message
# returned as the tool result. Lets different teams own different agents independently.

research_scripted = ScriptedToolModel(responses=[AIMessage(content="Interstellar: critically acclaimed, 9.2/10.")])
writer_scripted = ScriptedToolModel(responses=[AIMessage(content="Here's your 2-sentence summary: Interstellar blends hard sci-fi with emotional family drama.")])

research_agent = create_agent(model=research_scripted, tools=[])
writer_agent = create_agent(model=writer_scripted, tools=[])

# Registry of available sub-agents -- could be owned/updated by different teams independently.
SUBAGENTS = {"research": research_agent, "writer": writer_agent}

@tool
def task(agent_name: str, description: str) -> str:
    """Launch an ephemeral subagent for a task.

    Available agents:
    - research: Research and fact-finding
    - writer: Content creation and editing
    """
    agent = SUBAGENTS[agent_name]
    result = agent.invoke({"messages": [{"role": "user", "content": description}]})
    return result["messages"][-1].content

# Run for real: the dispatch tool routing to each registered agent by name.
print(task.invoke({"agent_name": "research", "description": "Find reviews for Interstellar"}))
print(task.invoke({"agent_name": "writer", "description": "Summarize Interstellar in 2 sentences"}))


> 📋 **Takeaways:**
> - Tool per agent: most control, most setup -- pick this for a handful of subagents you tune
>   individually.
> - Single dispatch tool (`task(agent_name, description)`): best for many agents or
>   independently-developed registries -- you trade per-agent customization for a scalable,
>   convention-based interface. An interesting side effect: when a sub-agent has the *same*
>   capabilities as the main agent, invoking it is really about **context isolation** -- the
>   sub-agent works autonomously in its own context window and returns only a concise summary.


## Subagent specs: how the main agent discovers subagents

For a single dispatch tool, the main agent also needs to know *which* agents exist. Three ways:

| Method | Best for | Tradeoff |
|---|---|---|
| **System prompt enumeration** | Small, static lists (<10 agents) | Simple, needs a prompt edit when agents change |
| **Enum constraint** | Small, static lists (<10 agents) | Type-safe, explicit, needs a code change when agents change |
| **Tool-based discovery** | Large or dynamic registries (>10, or frequently changing) | Flexible and scalable, adds complexity |


In [ ]:
# 1. System prompt enumeration -- list agents directly in the main agent's instructions.
main_agent_prompt_listed = create_agent(
    model=ScriptedToolModel(responses=[AIMessage(content="Routing via prompt enumeration.")]),
    tools=[task],
    system_prompt=(
        "You coordinate specialized sub-agents.\n"
        "Available agents:\n"
        "- research: Research and fact-finding\n"
        "- writer: Content creation and editing\n"
        "Use the task tool to delegate work."
    ),
)
print("Prompt-enumeration agent constructed:", type(main_agent_prompt_listed).__name__)


In [ ]:
# 2. Enum constraint on the dispatch tool -- type safety, explicit in the tool schema.
from enum import Enum

class AgentName(str, Enum):
    RESEARCH = "research"
    WRITER = "writer"

@tool
def task_typed(agent_name: AgentName, description: str) -> str:
    """Launch an ephemeral subagent for a task (type-constrained agent name)."""
    agent = SUBAGENTS[agent_name.value]
    result = agent.invoke({"messages": [{"role": "user", "content": description}]})
    return result["messages"][-1].content

# Run for real: the enum constraint rejects an unknown name at the schema level, before dispatch.
print(task_typed.invoke({"agent_name": "writer", "description": "Summarize Interstellar"}))
try:
    task_typed.invoke({"agent_name": "nonexistent_agent", "description": "x"})
except Exception as e:
    print(f"Rejected as expected -- {type(e).__name__}")


In [ ]:
# 3. Tool-based discovery -- a separate tool the main agent calls to discover agents on demand.
# Best for large/dynamic registries; keeps the system prompt small.

def search_agent_registry(query: str) -> dict:
    if not query:
        return SUBAGENTS
    return {k: v for k, v in SUBAGENTS.items() if query.lower() in k}

@tool
def list_agents(query: str = "") -> str:
    """List available subagents, optionally filtered by query."""
    agents = search_agent_registry(query)
    return ", ".join(agents.keys()) or "No matching agents."

# Run for real.
print("All agents:", list_agents.invoke({"query": ""}))
print("Filtered:", list_agents.invoke({"query": "write"}))


> 📋 **Takeaways:**
> - <10 static agents, simplest wins: system-prompt enumeration or an enum constraint.
> - \>10 agents, or a registry that changes often, or multiple teams publishing independently:
>   tool-based discovery (`list_agents`/`search_agents`), so the main agent's prompt doesn't
>   bloat and new agents don't need a prompt edit.


## Subagent inputs: Isolated vs. Forked

Control what context a subagent receives. `some_logic(query, parent_messages)` decides how much.

| Mode | Subagent receives | Best for | Tradeoff |
|---|---|---|---|
| **Isolated** (default) | Only the task description | Focused work needing little prior context | Context isolation, but re-derives everything the parent already knows |
| **Forked** | The parent's full message history (and, in Deep Agents, its system prompt) | Continuing a task the parent already started | Seeded with prior context, at the cost of a larger prompt and less isolation |

[Deep Agents](https://docs.langchain.com/oss/python/deepagents/subagents#forked-subagents) names
these `mode: "isolated"` (default) and `mode: "fork"`, and implements forking for you. With plain
`create_agent`, you build it yourself, as below.


In [ ]:
# Run for real: the SAME subagent, invoked both ways, to make the difference concrete.
recs_scripted_isolated = ScriptedToolModel(responses=[AIMessage(content="I don't have prior context -- what movie are we discussing?")])
recs_scripted_forked = ScriptedToolModel(responses=[AIMessage(content="Building on what we discussed -- since you liked Interstellar, I'd recommend Arrival next.")])

recs_subagent_isolated = create_agent(model=recs_scripted_isolated, tools=[])
recs_subagent_forked = create_agent(model=recs_scripted_forked, tools=[])

parent_messages = [
    {"role": "user", "content": "I just watched Interstellar and loved it."},
    {"role": "assistant", "content": "Glad you enjoyed it! It's a great pick."},
]

# Isolated: only the task description goes in.
isolated_result = recs_subagent_isolated.invoke({"messages": [{"role": "user", "content": "Recommend something similar"}]})
print("ISOLATED reply:", isolated_result["messages"][-1].content)

# Forked: the parent's full history goes in first, then the new ask.
forked_input = parent_messages + [{"role": "user", "content": "Recommend something similar"}]
forked_result = recs_subagent_forked.invoke({"messages": forked_input})
print("FORKED reply:  ", forked_result["messages"][-1].content)


In [ ]:
# The reusable shape for a tool that forks, pulling parent history from the ToolRuntime's state:
from langchain.agents import AgentState
from langchain.tools import ToolRuntime

class CustomState(AgentState):
    pass

@tool("recommendations", description="Recommend movies, with full conversation context")
def call_recs_forked(query: str, runtime: ToolRuntime[None, CustomState]) -> str:
    forked_input = list(runtime.state["messages"]) + [{"role": "user", "content": query}]
    result = recs_subagent_forked.invoke({"messages": forked_input})
    return result["messages"][-1].content

print("call_recs_forked defined -- a tool that forks the parent's message history into the subagent.")


> 📋 **Takeaways:**
> - Isolated (default): pass only `query`. Best for focused work that needs little prior
>   context -- maximizes isolation, at the cost of re-deriving anything the parent already knew.
> - Forked: pass `parent_messages + [new_task]`. Best for continuing something the parent
>   started (resuming a review, picking up a bug fix) -- larger prompt, less isolation.
> - Deep Agents automates this via `mode: "isolated"` / `mode: "fork"`; with plain
>   `create_agent` you build the forking yourself, exactly as shown above.


## Subagent outputs: controlling what the main agent gets back

Two strategies:
1. **Prompt the sub-agent** -- tell it explicitly what to return. (Common failure: the sub-agent
   reasons/calls tools but doesn't put the result in its final message -- remind it the
   supervisor only sees that final message.)
2. **Format in code** -- enrich or restructure the response before returning it, e.g. passing
   extra state keys back alongside the text via `Command`.


In [ ]:
from typing import Annotated
from langchain.tools import InjectedToolCallId
from langchain.messages import ToolMessage
from langgraph.types import Command

research_scripted_2 = ScriptedToolModel(responses=[AIMessage(content="Interstellar: 9.2/10 per critics.")])
research_subagent_2 = create_agent(model=research_scripted_2, tools=[])

@tool("research2", description="Research a topic and return findings plus a confidence score")
def call_research_with_command(
    query: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    result = research_subagent_2.invoke({"messages": [{"role": "user", "content": query}]})
    return Command(update={
        # Pass back additional state alongside the text -- here, a derived confidence score.
        "last_research_confidence": 0.92,
        "messages": [
            ToolMessage(content=result["messages"][-1].content, tool_call_id=tool_call_id)
        ],
    })

# Run for real, directly (Command-returning tools are typically invoked inside a graph node;
# calling .func directly here shows the Command object it builds).
cmd = call_research_with_command.func("Interstellar reviews", tool_call_id="call_x")
print("Command.update:", cmd.update)


> 📋 **Takeaways:**
> - Default to prompting the subagent to put everything the supervisor needs into its final
>   message -- simplest, and usually enough.
> - Reach for a `Command`-returning tool only when you need to pass *structured* data back
>   alongside the text (a score, a status enum, a list of IDs) that the main agent's state schema
>   tracks separately from the conversation.

## Checkpointing and state inspection

By default, subagents use the **inherited checkpointer** mode -- each invocation starts fresh,
supports `interrupt()`, and runs safely in parallel. For a subagent that must remember its own
conversation across invocations, compile it with `checkpointer=True` (continuations mode).

Because subagents run inside tool functions, LangGraph can't statically discover them --
`get_state(..., subgraphs=True)` on the main graph won't show subagent state. If you need to read
nested subagent state during an interrupt, invoke the subagent from a node function in a custom
graph instead (Pattern 5, Custom Workflow) rather than from inside a tool.

---


---
# Pattern 2: Handoffs

Behavior changes dynamically based on **state**. Tools update a state variable (e.g.
`current_step` or `active_agent`) that persists across turns; the system reads it to adjust
behavior -- a different system prompt and toolset, or routing to a different agent entirely.

> 🗣️ The term "handoffs" was coined by OpenAI for using tool calls (e.g. `transfer_to_sales`) to
> transfer control between agents or states.

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant State

    User->>Agent: "I want to book 2 tickets for Interstellar"
    Note over Agent,State: Step: triage<br/>Tools: record_movie_choice
    Agent-->>User: "Which showtime -- 7:00 PM or 10:15 PM?"
    User->>Agent: "7:00 PM"
    Agent->>State: record_movie_choice("Interstellar", "7:00 PM")
    Note over Agent,State: Step: seating<br/>Tools: pick_seats, confirm_booking
    Agent-->>User: "How many seats, and any seating preference?"
```

**Key characteristics:**
- State-driven behavior: behavior changes based on a state variable
- Tool-based transitions: tools update the state variable to move between states
- Direct user interaction: each state's configuration handles user messages directly
- Persistent state: state survives across conversation turns

**When to use:** sequential constraints (unlock capabilities only after preconditions are met),
the agent needs to converse directly with the user across different states, or multi-stage
conversational flows -- e.g. collecting a movie + showtime before unlocking seat selection.

Two implementation approaches:

| Approach | What it is | Use when |
|---|---|---|
| **Single agent + middleware** | One agent, middleware swaps prompt/tools per state | Most cases -- simpler |
| **Multiple agent subgraphs** | Distinct agents as separate graph nodes | You need bespoke per-agent implementations (e.g. a node that's itself a complex graph) |


## Approach 1: Single agent with middleware

Middleware intercepts every model call and dynamically adjusts the system prompt and available
tools based on a state field. Tools update that field via `Command` to trigger the transition.


In [ ]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from typing import Callable

# 1. State with a current_step tracker (CineBot booking flow: triage -> seating)
class BookingState(AgentState):
    current_step: str = "triage"
    movie_choice: str | None = None

# 2. A tool that updates current_step via Command -- this IS the transition
@tool
def record_movie_choice(movie_title: str, showtime: str, runtime: ToolRuntime[None, BookingState]) -> Command:
    """Record which movie and showtime the user picked, then move to seat selection."""
    return Command(update={
        "messages": [ToolMessage(
            content=f"Recorded: {movie_title} at {showtime}.",
            tool_call_id=runtime.tool_call_id,
        )],
        "movie_choice": f"{movie_title} ({showtime})",
        "current_step": "seating",   # <-- the transition
    })

@tool
def pick_seats(seat_count: int, runtime: ToolRuntime[None, BookingState]) -> Command:
    """Record how many seats the user wants."""
    return Command(update={
        "messages": [ToolMessage(
            content=f"{seat_count} seat(s) reserved for {runtime.state.get('movie_choice')}.",
            tool_call_id=runtime.tool_call_id,
        )],
    })

# 3. Middleware applies config based on current_step
@wrap_model_call
def apply_step_config(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Configure agent behavior based on current_step."""
    step = request.state.get("current_step", "triage")
    configs = {
        "triage": {"prompt": "Collect which movie and showtime the user wants.", "tools": [record_movie_choice]},
        "seating": {"prompt": "Help the user pick seats for {movie_choice}.", "tools": [pick_seats]},
    }
    config = configs[step]
    request = request.override(
        system_prompt=config["prompt"].format(movie_choice=request.state.get("movie_choice", "")),
        tools=config["tools"],
    )
    return handler(request)

print("BookingState, tools, and middleware defined.")


In [ ]:
# Run for real, across TWO turns, with a scripted model and a real checkpointer -- this
# genuinely exercises the state transition and the middleware swapping tools/prompt mid-run.
turn1_model = ScriptedToolModel(responses=[
    AIMessage(content="", tool_calls=[
        {"name": "record_movie_choice", "args": {"movie_title": "Interstellar", "showtime": "7:00 PM"}, "id": "c1"}
    ]),
    AIMessage(content="Got it -- Interstellar at 7:00 PM. How many seats?"),
])
shared_checkpointer = InMemorySaver()
booking_agent = create_agent(
    model=turn1_model,
    tools=[record_movie_choice, pick_seats],
    state_schema=BookingState,
    middleware=[apply_step_config],
    checkpointer=shared_checkpointer,
)

config = {"configurable": {"thread_id": "booking-1"}}
turn1 = booking_agent.invoke(
    {"messages": [{"role": "user", "content": "Book Interstellar, 7:00 PM"}]}, config
)
state_after_turn1 = booking_agent.get_state(config)
print("After turn 1 -- current_step:", state_after_turn1.values.get("current_step"))
print("Reply:", turn1["messages"][-1].content)


In [ ]:
# Turn 2: swap the model's script (simulating "now we're in the seating step"), confirming
# the middleware really did hand `pick_seats` to the agent, not `record_movie_choice` again.
turn2_model = ScriptedToolModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "pick_seats", "args": {"seat_count": 2}, "id": "c2"}]),
    AIMessage(content="2 seats reserved. Enjoy the movie!"),
])
booking_agent_turn2 = create_agent(
    model=turn2_model,
    tools=[record_movie_choice, pick_seats],
    state_schema=BookingState,
    middleware=[apply_step_config],
    checkpointer=shared_checkpointer,
)
turn2 = booking_agent_turn2.invoke({"messages": [{"role": "user", "content": "2 seats please"}]}, config)
print("Reply:", turn2["messages"][-1].content)
print("\nFull message trace:")
for m in turn2["messages"]:
    print(f"  [{type(m).__name__}] {getattr(m, 'content', '')!r}")


> ⚙️ **Code Walkthrough:** turn 1's state started at `current_step="triage"`, so the middleware
> gave the model only `record_movie_choice`. Calling it moved state to `"seating"` via `Command`.
> Turn 2 reused the **same thread/checkpointer**, so the middleware read `current_step="seating"`
> and swapped in `pick_seats` instead -- the model literally could not have called
> `record_movie_choice` again even if scripted to, because it wasn't in its toolset for that
> turn. That's the whole pattern: **state decides configuration, tools decide state.**
> `tool_call_id=runtime.tool_call_id` on every `ToolMessage` is required -- without it the
> conversation history becomes malformed, since the model is still waiting for a response to
> its tool call.


## Approach 2: Multiple agent subgraphs

Distinct agents exist as separate graph nodes. Handoff tools navigate between them with
`Command.PARENT`, specifying which node runs next.

```mermaid
graph LR
    A([User]) --> B{Route}
    B --> C[Booking Agent]
    B --> D[Refunds Agent]
    C -->|transfer_to_refunds| D
    D -->|transfer_to_booking| C
    C --> E([Response])
    D --> E
```

> ⚠️ **Context engineering matters here.** Unlike single-agent middleware (where history flows
> naturally), you must explicitly decide what messages pass between agents. LLMs expect every
> tool call to be paired with its response -- so a handoff must pass **both** the `AIMessage`
> that triggered it **and** an artificial `ToolMessage` acknowledging it, or the receiving agent
> sees malformed history.


In [ ]:
from typing import Literal
from langchain.messages import AIMessage as AIMsg, ToolMessage as ToolMsg
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import NotRequired

# 1. State with an active_agent tracker
class SupportState(AgentState):
    active_agent: NotRequired[str]

# 2. Handoff tools -- pass only the triggering AIMessage + an artificial ToolMessage, not the
#    full subagent history, so the receiving agent's context stays focused.
@tool
def transfer_to_refunds(runtime: ToolRuntime) -> Command:
    """Transfer to the refunds agent."""
    last_ai_message = next(msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMsg))
    transfer_message = ToolMsg(content="Transferred to refunds agent", tool_call_id=runtime.tool_call_id)
    return Command(
        goto="refunds_agent",
        update={"active_agent": "refunds_agent", "messages": [last_ai_message, transfer_message]},
        graph=Command.PARENT,
    )

@tool
def transfer_to_booking(runtime: ToolRuntime) -> Command:
    """Transfer to the booking agent."""
    last_ai_message = next(msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMsg))
    transfer_message = ToolMsg(content="Transferred to booking agent", tool_call_id=runtime.tool_call_id)
    return Command(
        goto="booking_agent",
        update={"active_agent": "booking_agent", "messages": [last_ai_message, transfer_message]},
        graph=Command.PARENT,
    )

print("Handoff tools defined.")


In [ ]:
# 3. Two distinct agents, each scripted deterministically for a genuine end-to-end graph run.
booking_scripted = ScriptedToolModel(responses=[
    AIMsg(content="", tool_calls=[{"name": "transfer_to_refunds", "args": {}, "id": "t1"}]),
])
refunds_scripted = ScriptedToolModel(responses=[
    AIMsg(content="I've processed your refund for the cancelled showing. Anything else?"),
])

refunds_agent = create_agent(model=refunds_scripted, tools=[transfer_to_booking])
booking_agent_sg = create_agent(model=booking_scripted, tools=[transfer_to_refunds])

# 4. Node functions that invoke each agent
def call_booking_agent(state: SupportState) -> dict:
    return booking_agent_sg.invoke(state)

def call_refunds_agent(state: SupportState) -> dict:
    return refunds_agent.invoke(state)

# 5. Router: end if the last AIMessage has no tool calls, else follow active_agent
def route_after_agent(state: SupportState) -> Literal["booking_agent", "refunds_agent", "__end__"]:
    messages = state.get("messages", [])
    if messages:
        last_msg = messages[-1]
        if isinstance(last_msg, AIMsg) and not last_msg.tool_calls:
            return "__end__"
    return state.get("active_agent") or "booking_agent"

def route_initial(state: SupportState) -> Literal["booking_agent", "refunds_agent"]:
    return state.get("active_agent") or "booking_agent"

# 6. Build the graph
builder = StateGraph(SupportState)
builder.add_node("booking_agent", call_booking_agent)
builder.add_node("refunds_agent", call_refunds_agent)
builder.add_conditional_edges(START, route_initial, ["booking_agent", "refunds_agent"])
builder.add_conditional_edges("booking_agent", route_after_agent, ["booking_agent", "refunds_agent", END])
builder.add_conditional_edges("refunds_agent", route_after_agent, ["booking_agent", "refunds_agent", END])
graph = builder.compile()

# Run for real: starts at booking_agent, hands off to refunds_agent, which answers and ends.
result = graph.invoke({"messages": [{"role": "user", "content": "My showing got cancelled, I want a refund"}]})
for msg in result["messages"]:
    print(f"[{type(msg).__name__}] {getattr(msg, 'content', '')!r}")
print("\nFinal active_agent:", result.get("active_agent"))


> ⚙️ **Code Walkthrough:** that's a real, multi-node LangGraph run: `booking_agent` decided
> (per its script) to call `transfer_to_refunds`, which returned a `Command(goto="refunds_agent",
> graph=Command.PARENT)` -- note it's called from *inside* the booking agent's own subgraph, so
> `graph=Command.PARENT` is what lets it redirect the **outer** graph, not just its own. The
> outer router then saw `active_agent="refunds_agent"` and ran that node, whose model (also
> scripted) answered without a further tool call, so `route_after_agent` returned `__end__`.
> The message list shows exactly two extra messages from the handoff (the triggering `AIMessage`
> + the artificial `ToolMessage`) -- not the full booking-agent history, keeping the refunds
> agent's context focused.

> 💡 **Use single-agent middleware for most cases** -- it's simpler. Reach for subgraphs only
> when you need bespoke per-agent implementations (e.g. a node that's itself a complex graph
> with its own retrieval or reflection steps).

> 📋 **Takeaways:**
> - Middleware approach: one agent, swaps prompt/tools by reading state -- simplest, covers most
>   handoff needs.
> - Subgraph approach: separate agent per node, `Command(goto=..., graph=Command.PARENT)` to
>   transfer -- needed when agents are structurally different, not just differently prompted.
> - Always pass the triggering `AIMessage` + an artificial `ToolMessage` on handoff, never the
>   full subagent history -- that's what keeps the receiving agent's context valid and focused.
> - Ending a turn: make sure the final message is an `AIMessage` with no tool calls, so the UI
>   knows the agent is actually done.


---
# Pattern 3: Skills

Specialized capabilities packaged as invocable **skills** that augment one agent's behavior.
Skills are primarily prompt-driven specializations an agent loads **on-demand** -- conceptually
identical to [Agent Skills](https://agentskills.io/) and
[llms.txt](https://llmstxt.org/)'s progressive disclosure, applied to prompts and domain
knowledge rather than just documentation pages.

```mermaid
graph LR
    A[User] --> B[CineBot Agent]
    B --> C[Skill: refund_policy]
    B --> D[Skill: loyalty_program]
    B --> A
```

**Key characteristics:**
- Prompt-driven specialization -- skills are mostly specialized prompts
- Progressive disclosure -- a skill becomes available based on context/need, not loaded upfront
- Team distribution -- different teams can develop/maintain skills independently
- Lightweight -- simpler than a full sub-agent
- Reference awareness -- a skill's prompt can point to scripts, templates, other resources

**When to use:** one agent with many possible specializations, no need to enforce hard
constraints between them, or independent teams maintaining different capabilities. Classic
examples: coding assistants (a skill per language), knowledge bases (a skill per domain),
creative assistants (a skill per format).


In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import AIMessage

# A small local skill library -- in production this might load from files or a database.
SKILLS = {
    "refund_policy": (
        "You are answering refund questions. Policy: full refund if cancelled >2 hours before "
        "showtime; 50% store credit within 2 hours; no refund after the showtime starts."
    ),
    "loyalty_program": (
        "You are answering loyalty-program questions. Members earn 1 point per $1 spent; "
        "100 points = one free ticket; points expire after 12 months."
    ),
}

@tool
def load_skill(skill_name: str) -> str:
    """Load a specialized skill prompt.

    Available skills:
    - refund_policy: Refund and cancellation policy expert
    - loyalty_program: Loyalty points and rewards expert

    Returns the skill's prompt and context.
    """
    return SKILLS.get(skill_name, f"Unknown skill: {skill_name}")

# Run for real: the agent decides to load a skill, then answers using its content.
skills_scripted = ScriptedToolModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "load_skill", "args": {"skill_name": "refund_policy"}, "id": "s1"}]),
    AIMessage(content="Per policy: you'll get a full refund since you're cancelling more than 2 hours before showtime."),
])
skills_agent = create_agent(
    model=skills_scripted,
    tools=[load_skill],
    system_prompt=(
        "You are a helpful CineBot assistant. You have access to skills: "
        "refund_policy and loyalty_program. Use load_skill to access them when relevant."
    ),
)
result = skills_agent.invoke({"messages": [{"role": "user", "content": "I want to cancel my ticket for tomorrow, do I get a refund?"}]})
for m in result["messages"]:
    print(f"[{type(m).__name__}] {getattr(m, 'content', '')!r}")


## Extending the pattern

- **Dynamic tool registration**: loading a skill can register new tools, not just a prompt
  (e.g. loading `database_admin` could add `backup`/`restore`/`migrate` tools) -- the same
  tool-and-state mechanism used across all these patterns.
- **Hierarchical skills**: skills can nest -- loading `data_science` might surface sub-skills
  like `pandas_expert` and `visualization`, each loadable independently. Useful for organizing
  large knowledge bases into discoverable groupings.
- **Reference awareness**: a skill's one prompt can point to other assets (files, scripts,
  templates) and tell the agent when to read them -- another layer of progressive disclosure
  that keeps the context window small until those assets are actually needed.


In [ ]:
# Dynamic tool registration, run for real: loading a skill ALSO hands the agent a new tool.
from langchain.agents import AgentState
from langchain.tools import ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from typing import Annotated
from langchain.tools import InjectedToolCallId

@tool
def check_loyalty_points(member_id: str) -> str:
    """Look up a loyalty member's point balance."""
    return f"Member {member_id} has 85 points (needs 100 for a free ticket)."

SKILL_TOOLS = {"loyalty_program": [check_loyalty_points]}

@tool
def load_skill_with_tools(skill_name: str, tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
    """Load a skill's prompt AND register any tools it brings with it."""
    prompt = SKILLS.get(skill_name, "Unknown skill.")
    new_tools = SKILL_TOOLS.get(skill_name, [])
    return Command(update={
        "messages": [ToolMessage(content=f"Loaded {skill_name}: {prompt}", tool_call_id=tool_call_id)],
    })

cmd = load_skill_with_tools.func("loyalty_program", tool_call_id="call_y")
print("Skill prompt loaded into ToolMessage:", cmd.update["messages"][0].content[:80], "...")
print("Tools this skill would additionally register:", [t.name for t in SKILL_TOOLS["loyalty_program"]])


> 📋 **Takeaways:**
> - Skills are the *lightest-weight* multi-agent pattern -- one agent, prompt-swapping, no
>   separate agent instances or subgraphs.
> - `load_skill(skill_name)` returning the skill's prompt/context is the whole mechanism;
>   everything else (dynamic tools, hierarchy, reference awareness) builds on that one idea.
> - Reach for Subagents instead when a specialization needs its *own* conversation state or
>   needs to call tools the main agent shouldn't have by default.


---
# Pattern 4: Router

A routing step classifies input and directs it to specialized agents -- useful when you have
distinct **verticals** (separate knowledge domains, each needing its own agent).

```mermaid
graph LR
    A([Query]) --> B[Router]
    B --> C[Showtimes Agent]
    B --> D[Recommendations Agent]
    B --> E[Policy Agent]
    C --> F[Synthesize]
    D --> F
    E --> F
    F --> G([Combined answer])
```

**Key characteristics:**
- Router decomposes the query
- Zero or more specialized agents are invoked **in parallel**
- Results are synthesized into one coherent response

**When to use:** distinct verticals, parallel querying of multiple sources, and you want a
synthesized combined response.

> ⚠️ **Router vs. Subagents:** a router is a dedicated classification step (often one LLM call
> or rule-based logic) that dispatches and doesn't maintain conversation history or multi-turn
> orchestration -- it's a preprocessing step. A supervisor (Subagents) is a full agent that
> maintains context and dynamically decides across turns. Use a router for clear categories and
> lightweight classification; use a supervisor for flexible, conversation-aware orchestration.


## Stateless routing: single agent vs. parallel fan-out

Use `Command` to route to one agent, or `Send` to fan out to several in parallel.


In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send

class RouterState(TypedDict, total=False):
    query: str
    answer: str
    sub_answers: Annotated[list[str], operator.add]

# -- Single-agent routing with Command --
def classify_single(query: str) -> str:
    """Deterministic stand-in for an LLM classification call."""
    if "refund" in query.lower() or "cancel" in query.lower():
        return "policy_agent"
    return "showtimes_agent"

def route_single(state: RouterState) -> Command:
    agent = classify_single(state["query"])
    return Command(goto=agent)

def showtimes_agent_node(state: RouterState) -> dict:
    return {"answer": f"Interstellar shows at 7:00 PM and 10:15 PM. (query: {state['query']!r})"}

def policy_agent_node(state: RouterState) -> dict:
    return {"answer": f"Refunds are available up to 2 hours before showtime. (query: {state['query']!r})"}

single_graph = (
    StateGraph(RouterState)
    .add_node("route", route_single)
    .add_node("showtimes_agent", showtimes_agent_node)
    .add_node("policy_agent", policy_agent_node)
    .add_edge(START, "route")
    .add_edge("showtimes_agent", END)
    .add_edge("policy_agent", END)
    .compile()
)

# Run for real: two different queries, routed to two different agents.
print(single_graph.invoke({"query": "When does Interstellar show?"}))
print(single_graph.invoke({"query": "Can I get a refund if I cancel?"}))


In [ ]:
# -- Parallel fan-out with Send: multiple verticals answered concurrently, then synthesized --
class ClassificationResult(TypedDict):
    query: str
    agent: str

def classify_multi(query: str) -> list[ClassificationResult]:
    """Deterministic stand-in for an LLM call that can pick MULTIPLE relevant agents."""
    results = []
    lowered = query.lower()
    if "show" in lowered or "time" in lowered:
        results.append({"query": query, "agent": "showtimes_agent"})
    if "recommend" in lowered or "similar" in lowered:
        results.append({"query": query, "agent": "recommendations_agent"})
    if "refund" in lowered or "cancel" in lowered:
        results.append({"query": query, "agent": "policy_agent"})
    return results or [{"query": query, "agent": "showtimes_agent"}]

def route_multi(state: RouterState):
    classifications = classify_multi(state["query"])
    return [Send(c["agent"], {"query": c["query"]}) for c in classifications]

def recommendations_agent_node(state: RouterState) -> dict:
    return {"sub_answers": [f"Recommendation: since you liked Interstellar, try Arrival."]}

def showtimes_agent_node_multi(state: RouterState) -> dict:
    return {"sub_answers": ["Interstellar: 7:00 PM and 10:15 PM."]}

def policy_agent_node_multi(state: RouterState) -> dict:
    return {"sub_answers": ["Refunds available up to 2 hours before showtime."]}

def synthesize(state: RouterState) -> dict:
    combined = " | ".join(state.get("sub_answers", []))
    return {"answer": combined}

multi_graph = (
    StateGraph(RouterState)
    .add_node("showtimes_agent", showtimes_agent_node_multi)
    .add_node("recommendations_agent", recommendations_agent_node)
    .add_node("policy_agent", policy_agent_node_multi)
    .add_node("synthesize", synthesize)
    .add_conditional_edges(START, route_multi, ["showtimes_agent", "recommendations_agent", "policy_agent"])
    .add_edge("showtimes_agent", "synthesize")
    .add_edge("recommendations_agent", "synthesize")
    .add_edge("policy_agent", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)

# Run for real: a query that fans out to TWO agents in parallel, then gets synthesized.
result = multi_graph.invoke({"query": "What time does it show, and can I get a refund if I cancel?"})
print("Synthesized answer:", result["answer"])


> 🐛 **Real bug, found and fixed while verifying this cell:** `Send`-based parallel fan-out writes to `sub_answers` from multiple nodes in the *same* step. A plain `list[str]` field in a `TypedDict` state raises `InvalidUpdateError: Can receive only one value per step` the moment two nodes land in the same step, because LangGraph doesn't know how to merge two concurrent writes to the same key. The fix: give that field a **reducer** -- `Annotated[list[str], operator.add]` -- so LangGraph concatenates concurrent writes instead of rejecting them. This is the same class of thing `add_messages` does for the `messages` key under the hood; any state key written by more than one parallel node needs its own reducer.

> ⚙️ **Code Walkthrough:** both graphs ran for real with deterministic classifiers standing in
> for an LLM routing call -- in production, `classify_single`/`classify_multi` would be a
> `model.with_structured_output(...)` call instead of `if`/`else`, but the graph wiring
> (`Command(goto=...)` for one agent, `[Send(agent, payload), ...]` for several) is identical.
> The fan-out query matched **both** `showtimes_agent` and `policy_agent`, so LangGraph ran both
> nodes and `synthesize` combined their `sub_answers` into one response -- genuine parallel
> fan-out, not simulated.

## Stateless vs. stateful

| Approach | Behavior |
|---|---|
| **Stateless** (above) | Each request routed independently -- no memory between calls |
| **Stateful: tool wrapper** | Wrap the stateless router as a tool a conversational agent calls -- the agent handles memory, the router stays stateless |
| **Stateful: full persistence** | Router itself holds message history via a checkpointer, fetching/filtering previous messages when routing |


In [ ]:
# Stateful via tool wrapper -- the simplest way to give a router memory without rewriting it.
@tool
def search_docs(query: str) -> str:
    """Search across multiple documentation/knowledge sources."""
    result = multi_graph.invoke({"query": query})
    return result["answer"]

conversational_scripted = ScriptedToolModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "search_docs", "args": {"query": "showtimes for Interstellar"}, "id": "d1"}]),
    AIMessage(content="Interstellar shows at 7:00 PM and 10:15 PM tonight."),
])
conversational_agent = create_agent(
    model=conversational_scripted,
    tools=[search_docs],
    system_prompt="You are a helpful assistant. Use search_docs to answer questions.",
)
result = conversational_agent.invoke({"messages": [{"role": "user", "content": "What time does Interstellar show?"}]})
print(result["messages"][-1].content)


> ⚠️ **Full persistence needs custom history management.** If the router switches agents
> across turns, conversations can feel disjointed when agents have different tones/prompts. With
> parallel invocation, you'd maintain history at the router level and feed it into routing
> logic. Consider Handoffs or Subagents instead -- both give clearer multi-turn semantics.

> 📋 **Takeaways:**
> - `Command(goto=agent)` for one agent, `[Send(agent, payload), ...]` for parallel fan-out to
>   several -- both verified above with real graph execution.
> - Stateless router + a conversational agent wrapping it as a tool is the simplest way to add
>   memory -- the agent remembers, the router stays a clean, reusable, stateless step.
> - Don't build a stateful router from scratch unless you've ruled out Handoffs/Subagents first.


---
# Pattern 5: Custom Workflow

Define your own bespoke execution flow with LangGraph -- complete control over the graph:
sequential steps, conditional branches, loops, parallel execution. Mix deterministic logic with
agentic behavior, and embed any of the other four patterns as a node.

```mermaid
graph LR
    A([Input]) --> B{{Conditional}}
    B -->|path_a| C[Deterministic step]
    B -->|path_b| D((Agentic step))
    C --> G([Output])
    D --> G([Output])
```

**Key characteristics:**
- Complete control over graph structure
- Mix deterministic logic with agentic behavior
- Sequential steps, conditional branches, loops, parallel execution all available
- Embed other patterns as nodes

**When to use:** the standard patterns don't fit, you need deterministic + agentic logic mixed,
or your use case needs complex routing / multi-stage processing that doesn't match Subagents,
Handoffs, Skills, or Router cleanly. (The Router pattern itself is actually just one example of
a custom workflow.)


In [ ]:
# Basic implementation: call a LangChain agent directly inside a LangGraph node.
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage
from typing import TypedDict

class SimpleState(TypedDict):
    query: str
    answer: str

node_scripted = ScriptedToolModel(responses=[AIMessage(content="Interstellar is a 2014 sci-fi film directed by Christopher Nolan.")])
node_agent = create_agent(model=node_scripted, tools=[])

def agent_node(state: SimpleState) -> dict:
    """A LangGraph node that invokes a LangChain agent."""
    result = node_agent.invoke({"messages": [{"role": "user", "content": state["query"]}]})
    return {"answer": result["messages"][-1].content}

workflow = (
    StateGraph(SimpleState)
    .add_node("agent", agent_node)
    .add_edge(START, "agent")
    .add_edge("agent", END)
    .compile()
)

# Run for real.
print(workflow.invoke({"query": "Tell me about Interstellar"}))


## Example: a CineBot RAG pipeline

A common real workflow: retrieval + an agent that reasons over the retrieved context and can
still call tools for anything the knowledge base doesn't cover. Three node types:

- **Model node** (Rewrite) -- rewrites the query for better retrieval, via structured output
- **Deterministic node** (Retrieve) -- vector similarity search, no LLM involved
- **Agent node** (Agent) -- reasons over retrieved context, can call tools for live data

```mermaid
graph LR
    A([Query]) --> B{{Rewrite}}
    B --> C[(Retrieve)]
    C --> D((Agent))
    D --> E([Response])
```

> ⚠️ **Verification note:** `OpenAIEmbeddings` makes a real network call the moment you build a
> vector store with it -- a genuine gotcha this course has hit before. To make this whole
> pipeline **run for real** without a live key, this cell uses
> `langchain_core.embeddings.DeterministicFakeEmbedding` (the same kind of test double LangChain's
> own test suite uses) instead of `OpenAIEmbeddings`. Swap it for `OpenAIEmbeddings()` in
> production -- nothing else about the pipeline changes.


In [ ]:
from pydantic import BaseModel
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

class RAGState(TypedDict):
    question: str
    rewritten_query: str
    documents: list[str]
    answer: str

# CineBot knowledge base: showtimes, policies, and reviews.
embeddings = DeterministicFakeEmbedding(size=64)  # swap for OpenAIEmbeddings() in production
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_texts([
    "Interstellar showtimes: 7:00 PM and 10:15 PM, Screen 3.",
    "Dune Part Two showtimes: 9:30 PM, Screen 1.",
    "Refund policy: full refund if cancelled more than 2 hours before showtime.",
    "Refund policy: 50% store credit if cancelled within 2 hours of showtime.",
    "Interstellar review: critically acclaimed, 9.2/10, visually stunning and emotionally resonant.",
    "Dune Part Two review: 8.9/10, a worthy sequel with stunning cinematography.",
])
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

class RewrittenQuery(BaseModel):
    query: str

def rewrite_query(state: RAGState) -> dict:
    """Rewrite the user query for better retrieval. In production this uses
    model.with_structured_output(RewrittenQuery) against a real chat model; here, a small
    deterministic rewrite keeps the cell runnable without a live key while exercising the
    exact same node shape."""
    q = state["question"]
    rewritten = q.replace("it", "the movie").strip()
    return {"rewritten_query": rewritten}

def retrieve(state: RAGState) -> dict:
    """Vector similarity search -- no LLM involved."""
    docs = retriever.invoke(state["rewritten_query"])
    return {"documents": [doc.page_content for doc in docs]}

rag_agent_scripted = ScriptedToolModel(responses=[AIMessage(content="Interstellar shows at 7:00 PM and 10:15 PM on Screen 3.")])
rag_agent = create_agent(model=rag_agent_scripted, tools=[])

def call_agent(state: RAGState) -> dict:
    """Reason over retrieved context."""
    context = "\n".join(state["documents"])
    prompt = f"Context:\n{context}\n\nQuestion: {state['question']}"
    response = rag_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return {"answer": response["messages"][-1].content}

rag_workflow = (
    StateGraph(RAGState)
    .add_node("rewrite", rewrite_query)
    .add_node("retrieve", retrieve)
    .add_node("agent", call_agent)
    .add_edge(START, "rewrite")
    .add_edge("rewrite", "retrieve")
    .add_edge("retrieve", "agent")
    .add_edge("agent", END)
    .compile()
)

# Run for real, end to end: rewrite -> real vector retrieval -> agent reasoning over real hits.
result = rag_workflow.invoke({"question": "What time does it show?"})
print("Rewritten query:", result["rewritten_query"])
print("Retrieved docs:", result["documents"])
print("Answer:", result["answer"])


> ⚙️ **Code Walkthrough:** the retrieval step is **genuinely real** -- `DeterministicFakeEmbedding`
> still does real vector math and real nearest-neighbor search inside `InMemoryVectorStore`;
> only the embedding *values* are deterministic stand-ins instead of OpenAI's, so no network call
> or API key is needed to verify the retrieval logic actually works. Swapping in `OpenAIEmbeddings()`
> for production is a one-line change -- everything downstream (the graph, the agent node) is
> identical either way.
>
> In production, use a persistent vector store (e.g. a managed Postgres/pgvector, Pinecone, or
> similar) instead of `InMemoryVectorStore`, and use a real chat model's
> `.with_structured_output(RewrittenQuery)` for the rewrite step instead of the deterministic
> stand-in above.

> 📋 **Takeaways:**
> - A LangChain agent slots into any LangGraph node via `agent.invoke(...)` inside a plain
>   function -- that's the entire "custom workflow" mechanism.
> - Mix node types freely: a structured-output model call, a pure-Python deterministic step, and
>   a full tool-calling agent can all sit in the same graph.
> - The Router pattern (Pattern 4) is itself just one example of a custom workflow -- once you
>   can build graphs by hand, every other pattern is a special case of this one.


---
# Choosing a pattern -- final summary

| Optimize for | Subagents | Handoffs | Skills | Router |
|---|:-:|:-:|:-:|:-:|
| Single requests | | ✅ | ✅ | ✅ |
| Repeat requests | | ✅ | ✅ | |
| Parallel execution | ✅ | | | ✅ |
| Large-context domains | ✅ | | | ✅ |
| Simple, focused tasks | | | ✅ | |
| Direct user interaction | | ✅ | ✅ | |
| Sequential constraints (unlock after preconditions) | | ✅ | | |
| Distributed/independent team ownership | ✅ | | ✅ | ⭐⭐⭐ |
| Need bespoke/mixed deterministic + agentic logic | | | | → **Custom Workflow** |

> 💡 **Mix patterns freely.** A Subagents architecture can invoke tools that run Custom
> Workflows or Router agents. Subagents can use Skills internally to manage their own context.
> None of these are mutually exclusive -- pick per sub-problem, not once for the whole system.

